# Advanced — Santander Customer Transaction: Bayesian Optimization

Dataset: Santander Customer Transaction Prediction  
Input files expected in `data/raw/`:

- `train.csv`
- `test.csv`

This notebook uses Bayesian Optimization with `BayesSearchCV` and compares it with `RandomizedSearchCV`.

In [ ]:
from pathlib import Path
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import randint, loguniform
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier

try:
    from skopt import BayesSearchCV
    from skopt.space import Integer, Real
except ImportError as exc:
    raise ImportError(
        "scikit-optimize is required for this notebook. Install it with: pip install scikit-optimize"
    ) from exc

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "output"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
SUBMISSION_DIR = OUTPUT_DIR / "submissions"

for d in [TABLE_DIR, FIGURE_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

if not train_path.exists():
    raise FileNotFoundError(f"Missing file: {train_path}")
if not test_path.exists():
    raise FileNotFoundError(f"Missing file: {test_path}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("First train columns:", train.columns[:10].tolist())

if "target" not in train.columns:
    raise ValueError(
        "The loaded train.csv has no 'target' column. "
        "Check that train.csv is the Santander training file, not test.csv or sample_submission.csv."
    )

feature_cols = [c for c in train.columns if c not in ["ID_code", "target"]]
X_full = train[feature_cols]
y_full = train["target"].astype(int)
X_test = test[[c for c in test.columns if c != "ID_code"]]

print("Features:", len(feature_cols))
print("Target distribution:")
print(y_full.value_counts(normalize=True))

In [ ]:
MAX_ROWS = 30000

if len(X_full) > MAX_ROWS:
    X_work, _, y_work, _ = train_test_split(
        X_full,
        y_full,
        train_size=MAX_ROWS,
        stratify=y_full,
        random_state=RANDOM_STATE,
    )
else:
    X_work = X_full.copy()
    y_work = y_full.copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X_work,
    y_work,
    test_size=0.25,
    stratify=y_work,
    random_state=RANDOM_STATE,
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

print("Working sample:", X_work.shape)
print("Train split:", X_train.shape)
print("Validation split:", X_valid.shape)

In [ ]:
base_model = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE,
    early_stopping=True,
    validation_fraction=0.1,
)

random_space = {
    "learning_rate": loguniform(0.01, 0.3),
    "max_iter": randint(80, 301),
    "max_leaf_nodes": randint(15, 65),
    "l2_regularization": loguniform(1e-6, 10),
}

start = time.time()
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=random_space,
    n_iter=12,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

random_search.fit(X_train, y_train)
random_runtime = time.time() - start

random_valid_pred = random_search.best_estimator_.predict_proba(X_valid)[:, 1]
random_valid_auc = roc_auc_score(y_valid, random_valid_pred)

print("RandomizedSearchCV best CV ROC-AUC:", random_search.best_score_)
print("RandomizedSearchCV validation ROC-AUC:", random_valid_auc)
print("RandomizedSearchCV runtime:", random_runtime)
print("RandomizedSearchCV best params:", random_search.best_params_)

In [ ]:
bayes_space = {
    "learning_rate": Real(0.01, 0.3, prior="log-uniform"),
    "max_iter": Integer(80, 300),
    "max_leaf_nodes": Integer(15, 64),
    "l2_regularization": Real(1e-6, 10, prior="log-uniform"),
}

start = time.time()
bayes_search = BayesSearchCV(
    estimator=base_model,
    search_spaces=bayes_space,
    n_iter=20,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

bayes_search.fit(X_train, y_train)
bayes_runtime = time.time() - start

bayes_valid_pred = bayes_search.best_estimator_.predict_proba(X_valid)[:, 1]
bayes_valid_auc = roc_auc_score(y_valid, bayes_valid_pred)

print("BayesSearchCV best CV ROC-AUC:", bayes_search.best_score_)
print("BayesSearchCV validation ROC-AUC:", bayes_valid_auc)
print("BayesSearchCV runtime:", bayes_runtime)
print("BayesSearchCV best params:", bayes_search.best_params_)

In [ ]:
advanced_results = pd.DataFrame([
    {
        "level": "advanced",
        "method": "RandomizedSearchCV",
        "dataset": "Santander Customer Transaction",
        "model": "HistGradientBoostingClassifier",
        "best_score_roc_auc": random_search.best_score_,
        "validation_roc_auc": random_valid_auc,
        "runtime_seconds": random_runtime,
        "best_params": str(random_search.best_params_),
    },
    {
        "level": "advanced",
        "method": "BayesSearchCV",
        "dataset": "Santander Customer Transaction",
        "model": "HistGradientBoostingClassifier",
        "best_score_roc_auc": bayes_search.best_score_,
        "validation_roc_auc": bayes_valid_auc,
        "runtime_seconds": bayes_runtime,
        "best_params": str(bayes_search.best_params_),
    },
])

advanced_results.to_csv(TABLE_DIR / "04_advanced_santander_bayes_random_results.csv", index=False)
advanced_results

In [ ]:
bayes_trials = pd.DataFrame(bayes_search.cv_results_)
bayes_trials.to_csv(TABLE_DIR / "05_advanced_santander_bayes_trials.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(bayes_trials) + 1),
    bayes_trials["mean_test_score"].cummax(),
    marker="o",
)
plt.xlabel("Bayesian optimization iteration")
plt.ylabel("Best CV ROC-AUC so far")
plt.title("Bayesian Optimization Progress — Santander")
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "03_advanced_bayes_progress.png", dpi=160)
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(advanced_results["method"], advanced_results["validation_roc_auc"])
plt.ylabel("Validation ROC-AUC")
plt.title("Advanced Method Comparison — Santander")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "04_advanced_method_comparison.png", dpi=160)
plt.show()

In [ ]:
best_model = bayes_search.best_estimator_
test_pred = best_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "ID_code": test["ID_code"] if "ID_code" in test.columns else np.arange(len(test_pred)),
    "target": test_pred,
})

submission.to_csv(SUBMISSION_DIR / "santander_bayes_submission_demo.csv", index=False)
submission.head()

## Short conclusion

Bayesian Optimization uses previous evaluations to choose more promising hyperparameter settings.  
Compared with RandomizedSearchCV, it does not sample completely blindly. It balances exploration and exploitation and is useful when model evaluations are expensive.